In [1]:
# %%
"""
Brightkite radius-of-gyration benchmark.

This script is organized as editor cells (``# %%``) so you can open it
in VS Code or Jupyter as a multi-cell workflow. It expects the Brightkite
CSV to be at `tests/shared/data/brightkite.csv`.

Cell 1: load dataset into Polars (full ~4M rows)
Cell 2: run the `skmob2` benchmark
Cell 3: run the original `skmob` benchmark (pandas conversion)
"""

import skmob2
import time
from pathlib import Path
import polars as pl
import pandas as pd

In [2]:
DATA_PATH = Path("../shared/data/loc-brightkite_totalCheckins.txt.gz")

if not DATA_PATH.exists():
    raise SystemExit(
        f"Dataset not found at {DATA_PATH}. Download or place the brightkite.csv file there."
    )

print("Reading CSV into Polars (this may take a while)...")
df = pd.read_csv(
    DATA_PATH,
    sep="\t",
    header=0,
    # nrows=100_000,
    names=["user", "check-in_time", "latitude", "longitude", "location id"],
)
print(f"Loaded dataframe with {len(df)} rows and columns: {list(df.columns)[:10]}")

Reading CSV into Polars (this may take a while)...
Loaded dataframe with 4747286 rows and columns: ['user', 'check-in_time', 'latitude', 'longitude', 'location id']


In [3]:
df = df.rename(columns={"latitude": "lat", "longitude": "lng", "check-in_time": "datetime"})
df["datetime"] = pd.to_datetime(df["datetime"])
df

,user,datetime,lat,lng,location id
0,0,2010-10-16 06:02:04+00:00,39.891383,-105.070814,7a0f88982aa015062b95e3b4843f9ca2
1,0,2010-10-16 03:48:54+00:00,39.891077,-105.068532,dd7cd3d264c2d063832db506fba8bf79
2,0,2010-10-14 18:25:51+00:00,39.750469,-104.999073,9848afcc62e500a01cf6fbf24b797732f8963683
3,0,2010-10-14 00:21:47+00:00,39.752713,-104.996337,2ef143e12038c870038df53e0478cefc
4,0,2010-10-13 23:31:51+00:00,39.752508,-104.996637,424eb3dd143292f9e013efa00486c907
...,...,...,...,...,...
4747281,58222,2009-01-23 02:30:34+00:00,33.833333,35.833333,9f6b83bca22411dd85460384f67fcdb0
4747282,58224,2009-01-03 15:06:54+00:00,33.833333,35.833333,9f6b83bca22411dd85460384f67fcdb0
4747283,58225,2009-01-20 13:58:14+00:00,33.833333,35.833333,9f6b83bca22411dd85460384f67fcdb0
4747284,58226,2009-01-20 13:30:09+00:00,33.833333,35.833333,9f6b83bca22411dd85460384f67fcdb0


In [4]:
skmob2.jump_lengths(df)

,user,jump_lengths
0,0,"[19.640494457447723, 0.0, 0.0, 1.7434335091684..."
1,1,"[6.505339409923645, 46.75443058363974, 53.9285..."
2,2,"[0.0, 0.0, 0.0, 0.0, 3.641014748771287, 0.0, 5..."
3,3,"[3861.275963494035, 4.061636923656222, 5.91633..."
4,4,"[15511.949011944975, 0.0, 15511.949011944975, ..."
...,...,...
51401,58222,[]
51402,58224,[]
51403,58225,[]
51404,58226,[]


In [5]:
import time
import tracemalloc
from skmob2.measures.spatial.radius_of_gyration import radius_of_gyration as rog_skmob2

print("\nWarming up skmob2...")
_ = skmob2.jump_lengths(df)

times = []
peak_memories = []

for i in range(5):
    time.sleep(0.5)
    
    # Start tracking memory allocations
    tracemalloc.start()
    
    start = time.perf_counter()
    _ = skmob2.jump_lengths(df)
    end = time.perf_counter()
    
    # Capture the peak memory used during the function call
    current_mem, peak_mem = tracemalloc.get_traced_memory()
    tracemalloc.stop() # Reset tracker for the next round
    
    duration = end - start
    peak_mem_mb = peak_mem / (1024 * 1024) # Convert bytes to Megabytes
    
    times.append(duration)
    peak_memories.append(peak_mem_mb)
    
    print(f"skmob2 Round {i+1}: {duration:.4f} seconds | Peak Overhead: {peak_mem_mb:.2f} MB")

print(f"skmob2 Average Time: {sum(times)/len(times):.4f} s")
print(f"skmob2 Minimum Time: {min(times):.4f} s")
print(f"skmob2 Average Peak Overhead: {sum(peak_memories)/len(peak_memories):.2f} MB\n")


Warming up skmob2...
skmob2 Round 1: 0.6748 seconds | Peak Overhead: 253.56 MB
skmob2 Round 2: 0.6900 seconds | Peak Overhead: 253.56 MB
skmob2 Round 3: 0.6580 seconds | Peak Overhead: 253.56 MB
skmob2 Round 4: 0.6655 seconds | Peak Overhead: 253.56 MB
skmob2 Round 5: 0.6846 seconds | Peak Overhead: 253.56 MB
skmob2 Average Time: 0.6746 s
skmob2 Minimum Time: 0.6580 s
skmob2 Average Peak Overhead: 253.56 MB



In [7]:
from skmob2.measures.spatial.radius_of_gyration import radius_of_gyration as rog_skmob2

print("\nWarming up skmob2...")
_ = skmob2.jump_lengths(df)

times = []
peak_memories = []

for i in range(5):
    time.sleep(0.5)
    
    start = time.perf_counter()
    _ = skmob2.jump_lengths(df)
    end = time.perf_counter()
    
    
    duration = end - start
    times.append(duration)
    
    print(f"skmob2 Round {i+1}: {duration:.4f} seconds")

print(f"skmob2 Average Time: {sum(times)/len(times):.4f} s")
print(f"skmob2 Minimum Time: {min(times):.4f} s")
# print(f"skmob2 Average Peak Overhead: {sum(peak_memories)/len(peak_memories):.2f} MB\n")


Warming up skmob2...
skmob2 Round 1: 1.1353 seconds
skmob2 Round 2: 1.1299 seconds
skmob2 Round 3: 1.1225 seconds
skmob2 Round 4: 1.1370 seconds
skmob2 Round 5: 1.1706 seconds
skmob2 Average Time: 1.1390 s
skmob2 Minimum Time: 1.1225 s


In [5]:
import time
import tracemalloc
from skmob2.measures.spatial.radius_of_gyration import radius_of_gyration as rog_skmob2

print("\nWarming up skmob2...")
_ = skmob2.jump_lengths(df)

times = []
peak_memories = []

for i in range(5):
    time.sleep(0.5)
    
    # Start tracking memory allocations
    tracemalloc.start()
    
    start = time.perf_counter()
    _ = skmob2.jump_lengths(df)
    end = time.perf_counter()
    
    # Capture the peak memory used during the function call
    current_mem, peak_mem = tracemalloc.get_traced_memory()
    tracemalloc.stop() # Reset tracker for the next round
    
    duration = end - start
    peak_mem_mb = peak_mem / (1024 * 1024) # Convert bytes to Megabytes
    
    times.append(duration)
    peak_memories.append(peak_mem_mb)
    
    print(f"skmob2 Round {i+1}: {duration:.4f} seconds | Peak Overhead: {peak_mem_mb:.2f} MB")

print(f"skmob2 Average Time: {sum(times)/len(times):.4f} s")
print(f"skmob2 Minimum Time: {min(times):.4f} s")
print(f"skmob2 Average Peak Overhead: {sum(peak_memories)/len(peak_memories):.2f} MB\n")


Warming up skmob2...
skmob2 Round 1: 4.3021 seconds | Peak Overhead: 368.98 MB
skmob2 Round 2: 4.2341 seconds | Peak Overhead: 368.97 MB
skmob2 Round 3: 4.3017 seconds | Peak Overhead: 368.97 MB
skmob2 Round 4: 4.2414 seconds | Peak Overhead: 368.97 MB
skmob2 Round 5: 4.2707 seconds | Peak Overhead: 368.97 MB
skmob2 Average Time: 4.2700 s
skmob2 Minimum Time: 4.2341 s
skmob2 Average Peak Overhead: 368.97 MB

